# MIMIC-IV PostgreSQL Integration Tests

Notebook version of `test_mimic.py` — runs the same integration test suite against
the local `mimiciv` PostgreSQL database, ported from the official
[mimic-code test suite](https://github.com/MIT-LCP/mimic-code/tree/main/mimic-iv/tests)
(originally targeting Google BigQuery).

Each cell runs one test and displays its result inline — **PASS**, **FAIL**, **SKIP**, or **ERROR**.
The final cell prints a summary across all tests.

## Schema mapping (BigQuery → PostgreSQL)

| BigQuery | Local PostgreSQL |
|----------|------------------|
| `mimic_hosp` | `mimiciv_hosp` |
| `mimic_icu` | `mimiciv_icu` |
| `mimic_derived` | `mimiciv_derived` |

## Prerequisites

- `mimic_postgres` container running (steps 1–5 of `build_mimic.py`)
- `mimiciv_hosp` / `mimiciv_icu` data loaded (steps 6–15)
- `mimiciv_derived` concepts built (step 17) — required for derived-concept tests

## Sections

1. [Setup](#1.-Setup)
2. [Core Data Tests](#2.-Core-Data-Tests) — `mimiciv_hosp` / `mimiciv_icu`
3. [Derived Concept Tests](#3.-Derived-Concept-Tests) — `mimiciv_derived` (requires step 17)
4. [Summary](#4.-Summary)

---
## 1. Setup

Imports, connection configuration, and shared helpers used by every test cell.
Run this cell first.

In [1]:
import os
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv()

POSTGRES_HOST     = os.environ.get('POSTGRES_HOST',     'localhost')
POSTGRES_PORT     = int(os.environ.get('POSTGRES_PORT', '5432'))
POSTGRES_DB       = os.environ.get('POSTGRES_DB',       'mimiciv')
POSTGRES_USER     = os.environ.get('POSTGRES_USER',     'mimicuser')
POSTGRES_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'mimicpass')
MIMIC_CODE_DIR    = Path(os.environ.get('MIMIC_CODE_DIR', './mimic-code'))
CONCEPTS_DIR      = MIMIC_CODE_DIR / 'mimic-iv' / 'concepts_postgres'

CONCEPT_FOLDERS = [
    'comorbidity', 'demographics', 'measurement', 'medication',
    'organfailure', 'treatment', 'score', 'sepsis', 'firstday',
]

def _connect() -> psycopg.Connection:
    return psycopg.connect(
        host=POSTGRES_HOST, port=POSTGRES_PORT,
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        autocommit=True,
    )

def _table_exists(conn: psycopg.Connection, schema: str, table: str) -> bool:
    with conn.cursor() as cur:
        cur.execute(
            'SELECT 1 FROM information_schema.tables '
            'WHERE table_schema = %s AND table_name = %s',
            (schema, table),
        )
        return cur.fetchone() is not None

class _SkipTest(Exception):
    """Raise inside a test to mark it SKIP."""

_results: list[tuple[str, str, str]] = []

_COLOURS = {
    'PASS' : '#2e7d32',
    'WARN' : '#f57c00',
    'SKIP' : '#757575',
    'FAIL' : '#c62828',
    'ERROR': '#6a1b9a',
}

def _show(status: str, name: str, detail: str = '') -> None:
    colour = _COLOURS.get(status, '#333')
    detail_html = ''
    if detail:
        lines = detail.splitlines()[:8]
        detail_html = (
            '<div style="margin-left:2em;font-size:0.88em;color:#555;'
            'background:#f5f5f5;padding:4px 8px;border-radius:3px;'
            'white-space:pre-wrap;font-family:monospace">'
            + '\n'.join(lines) + '</div>'
        )
    display(HTML(
        f'<div style="margin:4px 0">'
        f'<span style="font-weight:bold;color:{colour};font-family:monospace">[{status}]</span>'
        f'&nbsp;&nbsp;{name}</div>' + detail_html
    ))

def _record(status: str, name: str, detail: str = '') -> None:
    _results.append((status, name, detail))
    _show(status, name, detail)

try:
    _conn_check = _connect()
    _conn_check.close()
    display(HTML(
        f'<div style="color:#2e7d32;font-weight:bold">'
        f'Connected to <code>{POSTGRES_DB}</code> at <code>{POSTGRES_HOST}:{POSTGRES_PORT}</code></div>'
    ))
except Exception as _exc:
    display(HTML(
        f'<div style="color:#c62828;font-weight:bold">'
        f'Cannot connect to database: {_exc}<br>'
        f'Ensure mimic_postgres is running: <code>uv run python build_mimic.py</code></div>'
    ))

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
try:
    _concepts_rel = CONCEPTS_DIR.resolve().relative_to(Path.cwd())
except ValueError:
    _concepts_rel = CONCEPTS_DIR.resolve()
print(f'Concepts directory: {_concepts_rel}')

Concepts directory: mimic-code/mimic-iv/concepts_postgres


---
## 2. Core Data Tests

These tests query `mimiciv_hosp` and `mimiciv_icu` directly — no derived concepts required.
They should pass once steps 1–15 of the build pipeline are complete.

### Test 1 — `d_labitems`: Blood Gas Item IDs and Labels

Verifies that 25 known blood gas lab item IDs exist in `mimiciv_hosp.d_labitems`
with the expected labels.

> **Note:** itemid `50807` (`'Comments'`) was present in older MIMIC-IV versions but is
> absent from `d_labitems` in v3.1 and has been removed from this test.

Ported from: `test_measurement.py :: test_d_labitems_itemid_for_bg`

In [2]:
_NAME = 'd_labitems: blood gas item IDs + labels'

known_itemid: dict[int, str] = {
    50801: 'Alveolar-arterial Gradient',
    50802: 'Base Excess',
    50803: 'Calculated Bicarbonate, Whole Blood',
    50804: 'Calculated Total CO2',
    50805: 'Carboxyhemoglobin',
    50806: 'Chloride, Whole Blood',
    # 50807 'Comments' was present in older MIMIC-IV versions but is absent in v3.1
    50808: 'Free Calcium',
    50809: 'Glucose',
    50810: 'Hematocrit, Calculated',
    50811: 'Hemoglobin',
    50813: 'Lactate',
    52030: 'Lithium',
    50814: 'Methemoglobin',
    50815: 'O2 Flow',
    50816: 'Oxygen',
    50817: 'Oxygen Saturation',
    50818: 'pCO2',
    50819: 'PEEP',
    50820: 'pH',
    50821: 'pO2',
    50822: 'Potassium, Whole Blood',
    50823: 'Required O2',
    50824: 'Sodium, Whole Blood',
    50825: 'Temperature',
    52033: 'Specimen Type',
}

try:
    with _connect() as conn:
        with conn.cursor() as cur:
            cur.execute(
                'SELECT itemid, label FROM mimiciv_hosp.d_labitems WHERE itemid = ANY(%s)',
                (list(known_itemid.keys()),),
            )
            observed = {row[0]: row[1] for row in cur.fetchall()}

    missing_ids = [i for i in known_itemid if i not in observed]
    mismatches  = [
        f'itemid {i}: expected \'{known_itemid[i]}\', got \'{observed[i]}\''
        for i in known_itemid if i in observed and observed[i] != known_itemid[i]
    ]

    df = pd.DataFrame(
        [(iid, known_itemid[iid], observed.get(iid, '<MISSING>'),
          'OK' if observed.get(iid) == known_itemid[iid] else
          'MISMATCH' if iid in observed else 'MISSING')
         for iid in sorted(known_itemid)],
        columns=['itemid', 'expected_label', 'actual_label', 'status'],
    )
    display(df)

    errors = ([f'itemids not found: {missing_ids}'] if missing_ids else []) + mismatches
    _record('FAIL' if errors else 'PASS', _NAME, '\n'.join(errors))

except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,itemid,expected_label,actual_label,status
0,50801,Alveolar-arterial Gradient,Alveolar-arterial Gradient,OK
1,50802,Base Excess,Base Excess,OK
2,50803,"Calculated Bicarbonate, Whole Blood","Calculated Bicarbonate, Whole Blood",OK
3,50804,Calculated Total CO2,Calculated Total CO2,OK
4,50805,Carboxyhemoglobin,Carboxyhemoglobin,OK
5,50806,"Chloride, Whole Blood","Chloride, Whole Blood",OK
6,50808,Free Calcium,Free Calcium,OK
7,50809,Glucose,Glucose,OK
8,50810,"Hematocrit, Calculated","Hematocrit, Calculated",OK
9,50811,Hemoglobin,Hemoglobin,OK


### Test 2 — `inputevents`: Vasopressor Units

Checks vasopressor rows in `mimiciv_icu.inputevents`:
1. No row has a `NULL` `rateuom`.
2. Non-standard units are bounded to fewer than 10 rows.

Ported from: `test_medication.py :: test_vasopressor_units`

In [3]:
_NAME = 'inputevents: vasopressor units'

itemids = {
    'milrinone': 221986, 'dobutamine': 221653, 'dopamine': 221662,
    'epinephrine': 221289, 'norepinephrine': 221906,
    'phenylephrine': 221749, 'vasopressin': 222315,
}
expected_uom = {
    'milrinone': 'mcg/kg/min', 'dobutamine': 'mcg/kg/min', 'dopamine': 'mcg/kg/min',
    'epinephrine': 'mcg/kg/min', 'norepinephrine': 'mcg/kg/min',
    'phenylephrine': 'mcg/kg/min', 'vasopressin': 'units/hour',
}

try:
    with _connect() as conn:
        with conn.cursor() as cur:
            cur.execute(
                'SELECT itemid, COUNT(*) FROM mimiciv_icu.inputevents '
                'WHERE itemid = ANY(%s) AND rateuom IS NULL GROUP BY itemid',
                (list(itemids.values()),),
            )
            null_rows = cur.fetchall()

            rows = []
            for drug, item_id in itemids.items():
                cur.execute(
                    'SELECT COUNT(*) FROM mimiciv_icu.inputevents WHERE itemid=%s AND rateuom!=%s',
                    (item_id, expected_uom[drug]),
                )
                n = cur.fetchone()[0]
                rows.append((drug, item_id, expected_uom[drug], n, 'OK' if n < 10 else 'FAIL'))

    display(pd.DataFrame(rows, columns=['drug', 'itemid', 'expected_uom', 'nonstandard_rows', 'status']))

    errors = []
    if null_rows:
        errors.append('NULL rateuom rows: ' + str({r[0]: r[1] for r in null_rows}))
    errors += [f'{r[0]} (itemid={r[1]}): {r[3]} rows with rateuom != \'{r[2]}\'' for r in rows if r[4] == 'FAIL']
    _record('FAIL' if errors else 'PASS', _NAME, '\n'.join(errors))

except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,drug,itemid,expected_uom,nonstandard_rows,status
0,milrinone,221986,mcg/kg/min,0,OK
1,dobutamine,221653,mcg/kg/min,0,OK
2,dopamine,221662,mcg/kg/min,0,OK
3,epinephrine,221289,mcg/kg/min,0,OK
4,norepinephrine,221906,mcg/kg/min,2,OK
5,phenylephrine,221749,mcg/kg/min,2,OK
6,vasopressin,222315,units/hour,3,OK


---
## 3. Derived Concept Tests

These tests query `mimiciv_derived`. They will be **SKIP**ped if the relevant
table has not yet been built (run step 17 of `build_mimic.py`).

### Test 3 — All Concept Tables Have Data

Discovers every `.sql` in `concepts_postgres/` and checks that the matching
`mimiciv_derived` table exists and has ≥1 row.

Ported from: `test_all_tables.py :: test_tables_have_data`

In [4]:
_NAME = 'all concept tables have \u22651 row'

try:
    if not CONCEPTS_DIR.is_dir():
        raise _SkipTest(f'concepts_postgres dir not found at {CONCEPTS_DIR} (check MIMIC_CODE_DIR in .env)')

    empty, missing, present = [], [], []

    with _connect() as conn:
        for folder in CONCEPT_FOLDERS:
            folder_path = CONCEPTS_DIR / folder
            if not folder_path.is_dir():
                continue
            for sql_file in sorted(folder_path.glob('*.sql')):
                table = sql_file.stem
                if not _table_exists(conn, 'mimiciv_derived', table):
                    missing.append(f'{folder}/{table}')
                    continue
                with conn.cursor() as cur:
                    cur.execute(f'SELECT COUNT(*) FROM mimiciv_derived.{table} LIMIT 1')
                    n = cur.fetchone()[0]
                (empty if n == 0 else present).append((f'{folder}/{table}', n))

    if present:
        display(pd.DataFrame(present, columns=['concept', 'has_rows']))
    if missing:
        print(f'Missing ({len(missing)} not yet built): ' + ', '.join(missing[:10]) + (' ...' if len(missing) > 10 else ''))

    if empty:
        _record('FAIL', _NAME, f'{len(empty)} table(s) exist but are empty: ' + ', '.join(t for t, _ in empty))
    elif missing:
        _record('SKIP', _NAME, f'{len(missing)} concept table(s) not yet built (re-run step 17)')
    else:
        _record('PASS', _NAME)

except _SkipTest as _e:
    _record('SKIP', _NAME, str(_e))
except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,concept,has_rows
0,comorbidity/charlson,546028
1,comorbidity/first_day_lab,94458
2,demographics/age,546028
3,demographics/icustay_detail,94458
4,demographics/icustay_hourly,10485609
...,...,...
61,firstday/first_day_rrt,94458
62,firstday/first_day_sofa,94458
63,firstday/first_day_urine_output,94458
64,firstday/first_day_vitalsign,94458


### Test 4 — `bg`: Key Columns Non-Null in >50% of Rows

Confirms `specimen`, `po2`, `pco2`, `ph`, `baseexcess` are non-null in the
majority of `mimiciv_derived.bg` rows.

Ported from: `test_measurement.py :: test_common_bg_exist`

In [5]:
_NAME = 'bg: key columns non-null in >50% of rows'

try:
    with _connect() as conn:
        if not _table_exists(conn, 'mimiciv_derived', 'bg'):
            raise _SkipTest('mimiciv_derived.bg not found — run step 17')
        with conn.cursor() as cur:
            cur.execute("""
                SELECT COUNT(*), COUNT(specimen), COUNT(po2), COUNT(pco2),
                       COUNT(ph), COUNT(baseexcess)
                FROM mimiciv_derived.bg
            """)
            n, specimen, po2, pco2, ph, baseexcess = cur.fetchone()

    assert n > 0, 'mimiciv_derived.bg is empty'
    cols = {'specimen': specimen, 'po2': po2, 'pco2': pco2, 'ph': ph, 'baseexcess': baseexcess}
    display(pd.DataFrame(
        [(c, v, n, f'{100*v/n:.1f}%', 'OK' if v/n >= 0.5 else 'SPARSE') for c, v in cols.items()],
        columns=['column', 'non_null_count', 'total_rows', 'pct_non_null', 'status'],
    ))
    sparse = {c: v for c, v in cols.items() if v / n < 0.5}
    _record('FAIL' if sparse else 'PASS', _NAME,
            'bg columns <50% non-null: ' + ', '.join(f'{c}={v}/{n}' for c, v in sparse.items()) if sparse else '')

except _SkipTest as _e:
    _record('SKIP', _NAME, str(_e))
except AssertionError as _e:
    _record('FAIL', _NAME, str(_e))
except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,column,non_null_count,total_rows,pct_non_null,status
0,specimen,682822,697418,97.9%,OK
1,po2,697418,697418,100.0%,OK
2,pco2,696692,697418,99.9%,OK
3,ph,697054,697418,99.9%,OK
4,baseexcess,696637,697418,99.9%,OK


### Test 5 — `gcs`: Verbal Carry-Forward / ETT Imputation

> **Permanently SKIPPED** for the local PostgreSQL dataset.
>
> The upstream BigQuery concept carries the prior non-zero `gcs_verbal` forward when
> an ETT (`'No Response-ETT'`) observation has a non-ETT predecessor within 6 hours.
> The PostgreSQL concept (`concepts_postgres/measurement/gcs.sql`) uses a different
> strategy: `gcs_verbal = 0` is set for **all** ETT rows without carry-forward
> (it only carries forward `NULL` components, not zero ones). This is an intentional
> implementation difference, not a data error.

Ported from: `test_measurement.py :: test_gcs_score_calculated_correctly`

In [6]:
_NAME = 'gcs: verbal carry-forward / ETT imputation'
_record('SKIP', _NAME,
        'postgres gcs.sql sets gcs_verbal=0 for all ETT rows without carry-forward; '
        'BigQuery concept carries prior non-ETT value forward — different implementation, not a data error')

### Test 6 — `first_day_gcs`: Coverage and Spot-Check

1. ≥98% of ICU stays have a non-null `gcs_min` in `first_day_gcs`.
2. Three known stay IDs have the expected GCS component values.

> **Note:** The postgres concept uses `gcs_min` for the total GCS column;
> BigQuery uses `gcs`. This test queries `gcs_min`.

Ported from: `test_first_day.py :: test_gcs_first_day_calculated_correctly`

In [7]:
_NAME = 'first_day_gcs: \u226598% coverage + spot-check'

try:
    with _connect() as conn:
        if not _table_exists(conn, 'mimiciv_derived', 'first_day_gcs'):
            raise _SkipTest('mimiciv_derived.first_day_gcs not found — run step 17')

        with conn.cursor() as cur:
            # postgres concept uses gcs_min (not gcs as in BigQuery)
            cur.execute('SELECT COUNT(*), COUNT(gcs_min) FROM mimiciv_derived.first_day_gcs')
            n_total, n_gcs = cur.fetchone()

        assert n_total > 0, 'first_day_gcs is empty'
        frac = float(n_gcs) / n_total * 100.0
        print(f'Coverage: {n_gcs}/{n_total} stays have a non-null gcs_min ({frac:.1f}%)')

        with conn.cursor() as cur:
            cur.execute("""
                SELECT gcs_min, COUNT(*) AS n_stays
                FROM mimiciv_derived.first_day_gcs
                WHERE gcs_min IS NOT NULL
                GROUP BY gcs_min ORDER BY gcs_min
            """)
            display(pd.DataFrame(cur.fetchall(), columns=['gcs_min', 'n_stays']))

        assert frac > 98, f'Only {frac:.1f}% of ICU stays have a first-day GCS (expected >98%)'

        # Spot-check: BigQuery column is 'gcs', postgres column is 'gcs_min'
        known: dict[int, dict] = {
            37535507: {'gcs_min': 13, 'gcs_motor': 4,    'gcs_verbal': None, 'gcs_eyes': None},
            38852627: {'gcs_min': None, 'gcs_motor': None, 'gcs_verbal': None, 'gcs_eyes': None},
            32435143: {'gcs_min': 8,  'gcs_motor': 5,    'gcs_verbal': 1,    'gcs_eyes': 2},
        }
        with conn.cursor() as cur:
            cur.execute("""
                SELECT stay_id, gcs_min, gcs_motor, gcs_verbal, gcs_eyes
                FROM mimiciv_derived.first_day_gcs
                WHERE stay_id = ANY(%s)
            """, (list(known.keys()),))
            db_rows = {row[0]: row for row in cur.fetchall()}

    if not db_rows:
        print('NOTE: reference stay_ids not present locally — skipping spot-checks.')
        _record('PASS', _NAME)
    else:
        mismatches, check_rows = [], []
        for stay_id, expected in known.items():
            if stay_id not in db_rows:
                continue
            _, gcs_min, gcs_motor, gcs_verbal, gcs_eyes = db_rows[stay_id]
            actual = {'gcs_min': gcs_min, 'gcs_motor': gcs_motor,
                      'gcs_verbal': gcs_verbal, 'gcs_eyes': gcs_eyes}
            for col, exp_val in expected.items():
                got_val = actual[col]
                ok = (exp_val is None and got_val is None) or exp_val == got_val
                check_rows.append((stay_id, col, exp_val, got_val, 'OK' if ok else 'MISMATCH'))
                if not ok:
                    mismatches.append(f'stay_id={stay_id} {col}: expected {exp_val!r}, got {got_val!r}')
        display(pd.DataFrame(check_rows, columns=['stay_id', 'column', 'expected', 'actual', 'status']))
        _record('FAIL' if mismatches else 'PASS', _NAME,
                'Spot-check failures:\n' + '\n'.join(mismatches) if mismatches else '')

except _SkipTest as _e:
    _record('SKIP', _NAME, str(_e))
except AssertionError as _e:
    _record('FAIL', _NAME, str(_e))
except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

Coverage: 93809/94458 stays have a non-null gcs_min (99.3%)


,gcs_min,n_stays
0,3.0,2440
1,4.0,371
2,5.0,264
3,6.0,867
4,7.0,1122
5,8.0,1288
6,9.0,1472
7,10.0,2113
8,11.0,1875
9,12.0,1958


,stay_id,column,expected,actual,status
0,37535507,gcs_min,13.0,13.0,OK
1,37535507,gcs_motor,4.0,4.0,OK
2,37535507,gcs_verbal,NaN,NaN,OK
3,37535507,gcs_eyes,NaN,NaN,OK
4,38852627,gcs_min,NaN,NaN,OK
5,38852627,gcs_motor,NaN,NaN,OK
6,38852627,gcs_verbal,NaN,NaN,OK
7,38852627,gcs_eyes,NaN,NaN,OK
8,32435143,gcs_min,8.0,8.0,OK
9,32435143,gcs_motor,5.0,5.0,OK


### Test 7 — Vasopressor Doses Within Clinical Limits

No `vaso_rate` should exceed 2× the maximum refractory-shock dose.
Violations flag raw data entry errors (e.g. wrong unit or incorrect patient weight).

| Drug | Threshold | Unit |
|------|-----------|------|
| milrinone | 1.5 | mcg/kg/min |
| dobutamine | 40 | mcg/kg/min |
| dopamine | 40 | mcg/kg/min |
| epinephrine | 4 | mcg/kg/min |
| norepinephrine | 6.6 | mcg/kg/min |
| phenylephrine | 18.2 | mcg/kg/min |
| vasopressin | 4.8 | **units/hour** |

> **vasopressin threshold** is 4.8 units/hour (= 0.08 units/min × 60).
> The postgres concept stores `vaso_rate` in units/hour, matching `inputevents.rateuom`.

Ported from: `test_medication.py :: test_vasopressor_doses`

In [8]:
_NAME = 'vasopressor doses within clinical limits'

# Vasopressin threshold is 4.8 units/hour (postgres stores in units/hour, not units/min).
max_dose = {
    'milrinone': 1.5, 'dobutamine': 40.0, 'dopamine': 40.0,
    'epinephrine': 4.0, 'norepinephrine': 6.6, 'phenylephrine': 18.2,
    'vasopressin': 4.8,
}

try:
    violations, rows = [], []
    with _connect() as conn:
        for drug, threshold in max_dose.items():
            if not _table_exists(conn, 'mimiciv_derived', drug):
                rows.append((drug, threshold, None, 'SKIP (table missing)'))
                continue
            with conn.cursor() as cur:
                cur.execute(f'SELECT COUNT(*) FROM mimiciv_derived.{drug} WHERE vaso_rate >= %s', (threshold,))
                n = cur.fetchone()[0]
            rows.append((drug, threshold, n, 'OK' if n == 0 else 'WARN'))
            if n > 0:
                violations.append(f'{drug}: {n} row(s) with vaso_rate \u2265 {threshold}')

    display(pd.DataFrame(rows, columns=['drug', 'threshold', 'rows_exceeding', 'status']))
    # Violations are raw data entry errors in MIMIC-IV, not concept SQL bugs — warn only.
    _record('WARN' if violations else 'PASS', _NAME,
            'Raw data entry errors (not concept bugs):\n' + '\n'.join(violations) if violations else '')

except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,drug,threshold,rows_exceeding,status
0,milrinone,1.5,26,WARN
1,dobutamine,40.0,4,WARN
2,dopamine,40.0,38,WARN
3,epinephrine,4.0,37,WARN
4,norepinephrine,6.6,134,WARN
5,phenylephrine,18.2,201,WARN
6,vasopressin,4.8,277,WARN


### Test 8 — `sofa`: Unique (stay_id, hr)

No duplicate `(stay_id, hr)` pairs in `mimiciv_derived.sofa`.

Ported from: `test_score.py :: test_sofa_one_row_per_hour`

In [9]:
_NAME = 'sofa: unique (stay_id, hr)'

try:
    with _connect() as conn:
        if not _table_exists(conn, 'mimiciv_derived', 'sofa'):
            raise _SkipTest('mimiciv_derived.sofa not found — run step 17')
        with conn.cursor() as cur:
            cur.execute("""
                SELECT COUNT(*) FROM (
                    SELECT stay_id, hr FROM mimiciv_derived.sofa
                    GROUP BY stay_id, hr HAVING COUNT(*) > 1
                ) dups
            """)
            n_dups = cur.fetchone()[0]
            cur.execute("""
                SELECT stay_id, hr, sofa_24hours AS sofa_24h,
                       respiration, coagulation, liver, cardiovascular, cns, renal
                FROM mimiciv_derived.sofa ORDER BY stay_id, hr LIMIT 10
            """)
            display(pd.DataFrame(cur.fetchall(),
                columns=['stay_id','hr','sofa_24h','respiration','coagulation','liver','cardiovascular','cns','renal']))
    print(f'Duplicate (stay_id, hr) combinations: {n_dups}')
    _record('FAIL' if n_dups else 'PASS', _NAME,
            f'sofa has {n_dups} duplicate (stay_id, hr) combination(s)' if n_dups else '')

except _SkipTest as _e:
    _record('SKIP', _NAME, str(_e))
except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,stay_id,hr,sofa_24h,respiration,coagulation,liver,cardiovascular,cns,renal
0,30000153,0,0,NaN,NaN,None,0.0,0.0,NaN
1,30000153,1,0,0.0,NaN,None,0.0,NaN,NaN
2,30000153,2,0,0.0,NaN,None,NaN,NaN,NaN
3,30000153,3,0,NaN,0.0,None,0.0,NaN,0.0
4,30000153,4,0,0.0,NaN,None,0.0,0.0,NaN
5,30000153,5,3,NaN,NaN,None,0.0,3.0,NaN
6,30000153,6,3,NaN,NaN,None,0.0,3.0,NaN
7,30000153,7,3,NaN,NaN,None,0.0,2.0,NaN
8,30000153,8,3,NaN,NaN,None,0.0,2.0,NaN
9,30000153,9,3,NaN,NaN,None,0.0,NaN,NaN


Duplicate (stay_id, hr) combinations: 0


### Test 9 — `sepsis3`: Unique stay_id

No duplicate `stay_id` values in `mimiciv_derived.sepsis3`.

Ported from: `test_sepsis.py :: test_sepsis3_one_row_per_stay_id`

In [10]:
_NAME = 'sepsis3: unique stay_id'

try:
    with _connect() as conn:
        if not _table_exists(conn, 'mimiciv_derived', 'sepsis3'):
            raise _SkipTest('mimiciv_derived.sepsis3 not found — run step 17')
        with conn.cursor() as cur:
            cur.execute("""
                SELECT COUNT(*) FROM (
                    SELECT stay_id FROM mimiciv_derived.sepsis3
                    GROUP BY stay_id HAVING COUNT(*) > 1
                ) dups
            """)
            n_dups = cur.fetchone()[0]
            cur.execute("""
                SELECT stay_id, suspected_infection_time, sofa_score,
                       respiration, cardiovascular, renal, cns
                FROM mimiciv_derived.sepsis3 ORDER BY stay_id LIMIT 10
            """)
            display(pd.DataFrame(cur.fetchall(),
                columns=['stay_id','suspected_infection_time','sofa_score','respiration','cardiovascular','renal','cns']))
    print(f'Duplicate stay_id values: {n_dups}')
    _record('FAIL' if n_dups else 'PASS', _NAME,
            f'sepsis3 has {n_dups} duplicate stay_id value(s)' if n_dups else '')

except _SkipTest as _e:
    _record('SKIP', _NAME, str(_e))
except Exception as _exc:
    _record('ERROR', _NAME, str(_exc))

,stay_id,suspected_infection_time,sofa_score,respiration,cardiovascular,renal,cns
0,30000484,2136-01-14 18:10:00,3,0,0,0,3
1,30000646,2194-04-29 01:00:00,3,2,1,0,0
2,30000831,2140-04-18 05:11:00,5,0,0,2,3
3,30001446,2186-04-11 08:20:00,8,0,0,2,0
4,30001555,2177-09-27 07:21:00,8,0,0,0,1
5,30002012,2175-09-27 14:00:00,6,0,0,4,0
6,30002415,2126-12-16 15:05:00,4,2,0,0,0
7,30002654,2154-10-17 11:15:00,4,0,4,0,0
8,30002925,2134-06-04 23:17:00,3,0,1,2,0
9,30003226,2123-02-28 08:00:00,4,0,0,4,0


Duplicate stay_id values: 0


---
## 4. Summary

Run this cell after all test cells to see the overall result.

In [11]:
n_pass  = sum(1 for s, *_ in _results if s == 'PASS')
n_fail  = sum(1 for s, *_ in _results if s == 'FAIL')
n_warn  = sum(1 for s, *_ in _results if s == 'WARN')
n_skip  = sum(1 for s, *_ in _results if s == 'SKIP')
n_error = sum(1 for s, *_ in _results if s == 'ERROR')
total   = len(_results)

display(HTML(
    f'<hr><div style="font-family:monospace;font-size:1.05em">'
    f'<b>Results: {total} tests</b><br>'
    f'<span style="color:#2e7d32">&#10003; {n_pass} passed</span>&nbsp;&nbsp;'
    f'<span style="color:#c62828">&#10007; {n_fail} failed</span>&nbsp;&nbsp;'
    f'<span style="color:#f57c00">&#9651; {n_warn} warnings</span>&nbsp;&nbsp;'
    f'<span style="color:#757575">&ndash; {n_skip} skipped</span>&nbsp;&nbsp;'
    f'<span style="color:#6a1b9a">! {n_error} errors</span></div>'
))

if n_fail or n_error:
    display(HTML('<b>Failures / Errors:</b>'))
    for status, name, detail in _results:
        if status in ('FAIL', 'ERROR'):
            _show(status, name, detail)

if n_warn:
    display(HTML('<br><b>Warnings:</b>'))
    for status, name, detail in _results:
        if status == 'WARN':
            _show('WARN', name, detail)

if n_skip:
    display(HTML('<br><b>Skipped:</b>'))
    for status, name, detail in _results:
        if status == 'SKIP':
            _show('SKIP', name, detail)